In [70]:
import pandas as pd

In [71]:
march_clean = pd.read_csv("../data/march_clean.csv")
nov_clean = pd.read_csv("../data/nov_clean.csv")   # use of this was all exploratory, not used in final data
post_clean = pd.read_csv("../data/post_clean.csv")

**Merging March and November data sets**

In [72]:
def print_merge_diagnostics(left, right, merged, left_name, right_name):
    print(f"{left_name} rows before merge: {len(left)}")
    print(f"{right_name} rows before merge: {len(right)}")
    print(f"Rows after merge: {len(merged)}")

In [73]:
df = march_clean.merge(nov_clean, on="QKEY", how="inner")
df.shape
#df.head()

(8148, 13)

In [74]:
print_merge_diagnostics(march_clean, nov_clean, df, "March", "Wave 79")

March rows before merge: 8914
Wave 79 rows before merge: 12648
Rows after merge: 8148


In [75]:
# create binary variable for misinformation belief

march_clean["COVIDCREATE_W63.5"].value_counts()

df["covid_misinfo_belief"] = df["COVIDCREATE_W63.5"].map({"Came about naturally": 0,
                                                          "Was developed intentionally in a lab": 1,
                                                          "Doesn’t really exist": 1})
df["covid_misinfo_belief"].value_counts(dropna=False)

covid_misinfo_belief
0.0    4523
NaN    2136
1.0    1489
Name: count, dtype: int64

In [76]:
pd.crosstab(df["F_PARTYSUM_FINAL"], df["covid_misinfo_belief"], normalize="index")

covid_misinfo_belief,0.0,1.0
F_PARTYSUM_FINAL,,
DK/Refused/No lean,0.600000,0.400000
Dem/Lean Dem,0.850713,0.149287
Rep/Lean Rep,0.608190,0.391810


In [77]:
# merging March dataset with other November data set - ACTUAL USED DATA

df_new = march_clean.merge(post_clean, on="QKEY", how="inner")
df_new.head()
df_new.shape

(7806, 9)

In [78]:
print_merge_diagnostics(march_clean, post_clean, df_new, "March", "Wave 78")

March rows before merge: 8914
Wave 78 rows before merge: 11818
Rows after merge: 7806


In [79]:
# again create binary misinfo belief variable

df_new["covid_misinfo_belief"] = df_new["COVIDCREATE_W63.5"].map({"Came about naturally": 0,
                                                                  "Was developed intentionally in a lab": 1,
                                                                  "Doesn’t really exist": 1})

# and another for voted for Trump vs Biden

df_new["trump_vote"] = df_new["VOTEGEN_POST_W78"].map({"Joe Biden, the Democrat": 0,
                                                       "Donald Trump, the Republican": 1})

df_new[["covid_misinfo_belief", "trump_vote"]].dropna().shape

(5079, 2)

In [13]:
#df_new.to_csv("../data/merged_clean.csv", index=False) UPDATED AT END AFTER W23 ADDED

**Looking at Wave23 - 2016 post-election data set**

In [90]:
w23 = pd.read_csv("../data/w23.csv")

/var/folders/bg/q1vhbljx38x9h5cs27rldg_00000gp/T/ipykernel_5496/3908011183.py:1: DtypeWarning: Columns (14,15,30,31,32,33,46,48,51,52,55,56,57,58,59,60,61,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  w23 = pd.read_csv("../data/w23.csv")


In [91]:
## look for overlap to see if can compare respondents' 2020 vote to 2016 vote

w23["QKEY"].isin(df["QKEY"]).sum()

np.int64(2326)

In [92]:
# find presidential vote variable

#[col for col in w23.columns if "VOTE" in col.upper()]
for col in w23.columns:
    if "VOTE" in col.upper():
        print("\n", col)
        print(w23[col].value_counts(dropna=False))
#VOTEGENPOST_W23


 VOTED_W23
VOTED_W23
I definitely voted in the 2016 presidential election    3699
I did not vote in the 2016 presidential election         298
I planned to vote but wasn't able to                     173
Refused                                                   13
Name: count, dtype: int64

 VOTEGENPOST_W23
VOTEGENPOST_W23
Hillary Clinton, the Democrat                    1850
Donald Trump, the Republican                     1479
NaN                                               484
Gary Johnson, the Libertarian Party candidate     161
Voted for none/Other                              126
Jill Stein, the Green Party candidate              53
Refused                                            30
Name: count, dtype: int64

 VOTEDECTIME_W23
VOTEDECTIME_W23
Before September                     2432
NaN                                   640
In September                          360
In October                            344
Last few days before the election     254
The last week before the e

In [93]:
# make binary vote for Trump variable

w23["trump_vote_2016"] = w23["VOTEGENPOST_W23"].map({"Hillary Clinton, the Democrat": 0,
                                                     "Donald Trump, the Republican": 1})

w23["trump_vote_2016"].value_counts(dropna=False)

trump_vote_2016
0.0    1850
1.0    1479
NaN     854
Name: count, dtype: int64

In [94]:
# merge -> final data set
df = df_new.merge(w23[["QKEY", "trump_vote_2016"]], on="QKEY", how="left")

print(df.shape)
df.head()
df["trump_vote_2016"].value_counts(dropna=False)

(7806, 12)


trump_vote_2016
NaN    5872
0.0    1097
1.0     837
Name: count, dtype: int64

In [85]:
print_merge_diagnostics(df_new, w23, df, "df_new", "Wave 23")

df_new rows before merge: 7806
Wave 23 rows before merge: 4183
Rows after merge: 7806


In [86]:
pd.crosstab(df["trump_vote_2016"], df["trump_vote"], margins=True)

trump_vote,0.0,1.0,All
trump_vote_2016,,,
0.0,1045,13,1058
1.0,42,755,797
All,1087,768,1855


In [87]:
reg4 = df[["trump_vote", "covid_misinfo_belief", "F_PARTYSUM_FINAL", "F_AGECAT", "F_EDUCCAT", "trump_vote_2016"]].dropna()

reg4.shape #1407 complete data points **changes to 1406 in 03_regression

(1407, 6)

In [88]:
df.to_csv("../data/merged_clean.csv", index=False)